# CSE 242 — Assignment 2

**Name:** Muneeb Syed
**Student ID:** 2028396


## Question 1 — Maximum likelihood estimation and posterior (20 pts)

For the observed sequence $D=\mathrm{HHHTT}$, the likelihood is

$$
L(h;D)=p(D\mid h)=h^3(1-h)^2,\qquad 0\le h\le1.
$$

### (a) MLE (10 pts)

Maximizing the log-likelihood $3\log h+2\log(1-h)$ gives

$$
\frac{3}{h}-\frac{2}{1-h}=0
\quad\Longrightarrow\quad
3(1-h)=2h
\quad\Longrightarrow\quad
\boxed{\hat h_{\rm MLE}=\frac{3}{5}=0.6}.
$$

### (b) Posterior (10 pts)

By Bayes' rule, for any prior density $p(h)$ on $[0,1]$,

$$
p(h\mid D)=
\frac{h^3(1-h)^2p(h)}
{\displaystyle\int_0^1 u^3(1-u)^2p(u)\,du},
\qquad 0\le h\le1.
$$

The denominator is the normalizing constant: the marginal probability of the observed data.

## Question 2 — Linear regression (40 pts)

The code fixes `random_state=42`, so the train/test split and reported results are reproducible. Spending and sales are both measured in thousands, so each coefficient is the expected change in thousands of items sold for an additional thousand dollars spent, holding the other included predictors fixed.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Works whether the notebook is run from Assignment2/ or its parent directory.
data_path = Path('Advertising.csv')
if not data_path.exists():
    data_path = Path('Assignment2') / 'Advertising.csv'
data = pd.read_csv(data_path, index_col=0)
data.head()


### Question 2.1 — Split the data (5 pts)


In [ ]:
X = data[['TV', 'Radio', 'Newspaper']]
y = data['Sales']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print('X_train:', X_train.shape, 'y_train:', y_train.shape)
print('X_test: ', X_test.shape, 'y_test: ', y_test.shape)


### Question 2.2 — Train the model (10 pts)


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept:', model.intercept_)
print(pd.Series(model.coef_, index=X.columns, name='coefficient'))


### Question 2.3 — Interpret coefficients (10 pts)


In [ ]:
equation = (
    f'Sales = {model.intercept_:.4f} '
    f'+ ({model.coef_[0]:.4f})·TV '
    f'+ ({model.coef_[1]:.4f})·Radio '
    f'+ ({model.coef_[2]:.4f})·Newspaper'
)
print(equation)
print(
    f'\nInterpretation of TV: holding Radio and Newspaper spending fixed, an additional $1,000 spent on TV is associated with an estimated {model.coef_[0]:.4f} thousand-unit change in Sales (about {model.coef_[0] * 1000:.1f} items).'
)


The printed equation is the trained model for the fixed split. The TV interpretation is conditional: it describes TV's estimated association with sales **while Radio and Newspaper are held fixed**; it does not by itself prove causation.


### Question 2.4 — RMSE on the test set (5 pts)


In [ ]:
y_pred = model.predict(X_test)
rmse_all = mean_squared_error(y_test, y_pred) ** 0.5
print(f'Test RMSE using TV, Radio, and Newspaper: {rmse_all:.4f}')


### Question 2.5 — Does Newspaper help? (10 pts)


In [ ]:
X_reduced = data[['TV', 'Radio']]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reduced, y, test_size=0.20, random_state=42
)

reduced_model = LinearRegression().fit(Xr_train, yr_train)
reduced_pred = reduced_model.predict(Xr_test)
rmse_reduced = mean_squared_error(yr_test, reduced_pred) ** 0.5

print(f'RMSE, TV + Radio + Newspaper: {rmse_all:.4f}')
print(f'RMSE, TV + Radio:             {rmse_reduced:.4f}')
print(f'Difference (reduced − full):  {rmse_reduced - rmse_all:.4f}')
if rmse_reduced < rmse_all:
    print('Observation: on this test split, removing Newspaper improves RMSE slightly.')
elif rmse_reduced > rmse_all:
    print('Observation: on this test split, Newspaper improves RMSE slightly.')
else:
    print('Observation: the two models have identical RMSE on this test split.')


The comparison is fair because the same `random_state` gives both models the same observations in their train and test partitions. A lower RMSE is better. The output describes predictive performance on this particular held-out split; a small difference should be checked with cross-validation before making a broad conclusion.


## Question 3 — Principal component analysis (40 pts)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

sns.set_theme()
digits = load_digits()
X_digits = digits.data
print('Digit feature matrix:', X_digits.shape)  # 1797 images × 64 pixels


### Question 3.1 — Reduce to two components (10 pts)


In [ ]:
pca_2 = PCA(n_components=2)
projected = pca_2.fit_transform(X_digits)

print('Original shape:', X_digits.shape)
print('Projected shape:', projected.shape)

plt.figure(figsize=(8, 6))
scatter = plt.scatter(projected[:, 0], projected[:, 1], c=digits.target,
                      cmap='tab10', alpha=0.7, edgecolors='none')
plt.xlabel('First principal component')
plt.ylabel('Second principal component')
plt.colorbar(scatter, ticks=range(10), label='Digit label')
plt.title('Digits projected onto the first two principal components')
plt.show()


### Question 3.2 — Explained variance by component (15 pts)


In [ ]:
pca_all = PCA()
pca_all.fit(X_digits)
explained_ratio = pca_all.explained_variance_ratio_

print('Explained-variance ratio for each component:')
for i, ratio in enumerate(explained_ratio, start=1):
    print(f'Component {i:2d}: {ratio:.6f}')

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(explained_ratio) + 1), explained_ratio, marker='o', ms=3)
plt.xlabel('i-th component')
plt.ylabel('Ratio of explained variance')
plt.title('Explained variance ratio of each PCA component')
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(range(1, len(explained_ratio) + 1), np.cumsum(explained_ratio), marker='o', ms=3)
plt.axhline(0.80, color='crimson', ls='--', label='80% variance')
plt.xlabel('Number of components retained')
plt.ylabel('Cumulative explained variance')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


### Question 3.3 — Observations about eigenvectors (5 pts)

Each eigenvector can be reshaped into an 8×8 **eigen-digit**. The early components contain smooth, large-scale stroke patterns: they capture the major ways digit images vary, such as overall brightness, location, width, and prominent loops or vertical strokes. Later components look progressively more detailed and less visually structured because each accounts for much less remaining variance. The eigenvectors are not themselves typical digit images; they are orthogonal directions of variation learned from all digit images.


In [ ]:
fig, axes = plt.subplots(6, 12, figsize=(12, 7))
for index, ax in enumerate(axes.flat):
    if index < pca_all.components_.shape[0]:
        ax.imshow(pca_all.components_[index].reshape(8, 8), cmap='gray')
        ax.set_title(f'{index + 1}')
    ax.axis('off')
fig.suptitle('PCA eigenvectors (eigen-digits)')
plt.tight_layout()
plt.show()


### Question 3.4 — PCA reconstruction of noisy images (10 pts)


In [ ]:
rng = np.random.default_rng(42)
noisy_imgs = X_digits + rng.normal(loc=0, scale=5, size=X_digits.shape)

# Fit a PCA model that retains enough components to explain 80% of the
# variance in the noisy images, then reconstruct the images from that space.
pca_noise = PCA(n_components=0.80)
lower_dimension_noise = pca_noise.fit_transform(noisy_imgs)
reconstructed_noise = pca_noise.inverse_transform(lower_dimension_noise)

print('Components retained for 80% noisy-data variance:', pca_noise.n_components_)

fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for i in range(10):
    axes[0, i].imshow(noisy_imgs[i].reshape(8, 8), cmap='gray')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructed_noise[i].reshape(8, 8), cmap='gray')
    axes[1, i].axis('off')
axes[0, 0].set_ylabel('Noisy', rotation=0, labelpad=28)
axes[1, 0].set_ylabel('PCA\nreconstructed', rotation=0, labelpad=28)
plt.tight_layout()
plt.show()


PCA can make the reconstructed images cleaner because it projects each image onto a lower-dimensional subspace containing the dominant, shared structure of the data. Random pixel noise is comparatively uncorrelated and tends to lie in many low-variance directions. When only enough principal components to retain 80% of the variance are kept, much of that low-variance detail is discarded during projection; the inverse transform therefore restores the main digit structure but not all of the noise. This is a trade-off: aggressive dimensionality reduction can also remove genuine fine detail.



I used AI for minor assistance and understanding help: https://chatgpt.com/share/6adb0a48-f61c-83e8-a0ff-4ceba1827c09